In [2]:
import torch
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling, TrainerCallback
from datasets import load_dataset
import logging
import os

os.environ["NCCL_P2P_DISABLE"] = "1"
os.environ["NCCL_IB_DISABLE"] = "1"


# Configure logging
logging.basicConfig(format='%(asctime)s - %(levelname)s - %(message)s', level=logging.INFO)
logger = logging.getLogger(__name__)

# ---------------------------
# 1. Model Initialization (from scratch)
# ---------------------------
# We use the configuration of a popular small-scale LLM (facebook/opt-125m)
# but initialize the model randomly (i.e. train from scratch)
model_name = "facebook/opt-125m"
model_name = "EleutherAI/gpt-neo-125M"
config = AutoConfig.from_pretrained(model_name)  # load config; do NOT load pretrained weights
model = AutoModelForCausalLM.from_config(config)   # randomly initialized model

# Load tokenizer (we can reuse the pretrained tokenizer)
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token  # assign PAD token if missing

# Enable gradient checkpointing to reduce memory usage (at the cost of additional compute)
model.gradient_checkpointing_enable()
logger.info(f"Initialized model from scratch with configuration from '{model_name}'.")

# ---------------------------
# 2. Dataset Preparation
# ---------------------------
# Load the Wikitext-2 dataset as our general-purpose text corpus.
# For a quick profiling run, we use only a small subset.
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
train_dataset = dataset["train"].select(range(1000))  # limit to 1000 examples for this demo

# Tokenization: convert text to token IDs (truncated to a maximum length)
def tokenize_function(example):
    return tokenizer(example["text"], truncation=True, max_length=128)

tokenized_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# Data collator: handles padding and prepares labels for causal LM (labels equal to input_ids)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# ---------------------------
# 3. Trainer Setup with Memory Profiling Callback
# ---------------------------
# TrainingArguments are set to run only 3 steps and use a small batch size suitable for an 8–12GB GPU.
training_args = TrainingArguments(
    output_dir="output",
    per_device_train_batch_size=2,
    max_steps=3,  # run only 3 training iterations for profiling
    gradient_accumulation_steps=1,
    fp16=False,  # using full precision; set True if your GPU supports mixed precision to save memory
    logging_steps=1,
    report_to=[],  # disable external logging (e.g., wandb)
    disable_tqdm=False,
    use_cpu=True
)

# Custom callback to log GPU memory usage at the start and end of each step
class MemoryProfilerCallback(TrainerCallback):
    def on_step_begin(self, args, state, control, **kwargs):
        torch.cuda.reset_peak_memory_stats()  # reset peak memory stats at beginning of each step

    def on_step_end(self, args, state, control, **kwargs):
        alloc = torch.cuda.memory_allocated()
        reserved = torch.cuda.memory_reserved()
        peak = torch.cuda.max_memory_allocated()
        step = state.global_step
        logger.info(f"Step {step}: allocated={alloc/1024**2:.2f} MB, reserved={reserved/1024**2:.2f} MB, peak={peak/1024**2:.2f} MB")
        return control

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
    callbacks=[]
)

# ---------------------------
# 4. Training with PyTorch Profiler
# ---------------------------
# We set up the PyTorch Profiler to capture CPU and CUDA activities with memory profiling enabled.
activities = [torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA]
activities = [torch.profiler.ProfilerActivity.CPU]
profiler = torch.profiler.profile(
    activities=activities,
    profile_memory=True,  # capture tensor memory allocations
    record_shapes=True,    # record shapes of tensors for deeper insight,
    with_stack=True,
    with_modules=True,
    on_trace_ready=torch.profiler.tensorboard_trace_handler(
        "./output/tb_profiler"
    )
)

profiler.start()
trainer.train()  # run 3 training iterations
profiler.stop()

# Print a summary of the profiler's memory usage by CUDA operation (top 10 ops)
print("Profiler Memory Usage Summary (top CUDA ops):")
print(profiler.key_averages().table(sort_by="self_cuda_memory_usage", row_limit=10))


2025-03-04 13:56:05,307 - INFO - Initialized model from scratch with configuration from 'EleutherAI/gpt-neo-125M'.


Step,Training Loss
1,10.862800
2,9.407000
3,10.138600


Profiler Memory Usage Summary (top CUDA ops):
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                               aten::to         0.30%       4.291ms         1.58%      22.843ms       5.368us      82.46 Kb           0 b          4255  
                aten::_has_compatible_shallow_copy_type         0.00%      18.000us         0.00%      18.000us       0.056us           0 b           0 b           322  
                                            aten::empty         0.17%       2.495ms         0.17%       

In [10]:
from ures.files import filter_files
from perf_estimator.xmem import XMem
from pathlib import Path


profiler_files = filter_files(
    "pt.trace.json", str(Path().home().joinpath("OneDrive - University of Glasgow/Documents/100-code/100-xMem/playground/output")), fuzz=True
)[-1]

xmen = XMem(
    batch_size=100,
    max_gpu_memory_in_gb=8,
)
result = xmen.estimate(profiler_file=profiler_files, trainer_enable=True)
result


2025-03-04 14:10:05,855 - INFO - New block created: 8589934592


AttributeError: 'NoneType' object has no attribute 'get_memory_activities'